# 모듈 02: 프롬프트 템플릿

모듈 01에서 AI와 직접 대화하는 법을 배웠습니다. 이번에는 **재사용 가능한 프롬프트 틀**을 만들어 봅니다.

---

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | ChatPromptTemplate 기초 - 변수 있는 프롬프트 틀 만들기 |
| 2 | 같은 템플릿을 다른 변수로 재사용하기 |
| 3 | partial()로 변수 일부를 미리 고정하기 |
| 4 | FewShotChatMessagePromptTemplate - 예시를 주면 패턴을 따라옴 |
| 5 | 실전: 번역기 만들기 |

**예상 소요 시간**: 약 45분

> **전제 조건**: 모듈 01이 완료되어 `.env` 파일에 `GOOGLE_API_KEY`가 설정되어 있어야 합니다.

## 학습 목표

이 모듈을 완료하면 다음 세 가지를 할 수 있습니다:

1. **ChatPromptTemplate로 재사용 가능한 프롬프트 만들기**: `{변수명}` 문법으로 틀을 만들고, 변수를 채워 프롬프트를 완성하는 방법을 압니다.

2. **partial()로 일부 변수 미리 고정하기**: 자주 쓰는 변수를 미리 고정해 두고, 나머지 변수만 바꿔서 쓰는 방법을 압니다.

3. **FewShotChatMessagePromptTemplate 활용하기**: 몇 가지 예시를 보여주면 AI가 그 패턴을 따라 답하도록 프롬프트를 구성하는 방법을 압니다.

## 전체 흐름도

이 노트북의 전체 흐름을 한눈에 보면 이렇습니다:

```
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  [1] .env 로드 + llm 초기화                                │
│       ↓                                                     │
│  [2] ChatPromptTemplate 기본 사용법                         │
│       → 변수 채우기 → 메시지 확인 → llm 호출               │
│       ↓                                                     │
│  [3] 같은 템플릿 재사용 (Python / JavaScript / Java)        │
│       ↓                                                     │
│  [4] partial()로 language 고정 → topic만 바꾸기             │
│       ↓                                                     │
│  [5] FewShotChatMessagePromptTemplate                       │
│       → examples 정의 → 감정 분류기 실행                   │
│       ↓                                                     │
│  [6] 실전 응용: 번역기 (partial 복습)                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> **팁**: 셀을 순서대로 실행하세요! 위에서 아래로 하나씩 `Shift + Enter`를 눌러 실행하면 됩니다.

---

## 섹션 2: 개념 소개

### 프롬프트 템플릿이란?

**요리 레시피 비유**

볶음밥을 만들 때마다 레시피를 처음부터 새로 쓴다면 어떨까요? 재료 이름만 달라질 뿐 나머지는 똑같은데도 매번 전부 다시 써야 합니다.

프롬프트 템플릿은 이 레시피에서 **재료 부분만 빈칸**으로 남겨두는 것입니다:

| 방법 | 예시 |
|------|------|
| 매번 새로 쓰기 | "당신은 Python 전문가입니다. 리스트에 대해 설명해주세요." |
| 매번 새로 쓰기 | "당신은 JavaScript 전문가입니다. 클로저에 대해 설명해주세요." |
| **템플릿 재사용** | "당신은 **{language}** 전문가입니다. **{topic}**에 대해 설명해주세요." |

템플릿을 한 번 만들어두면 `language`와 `topic`만 바꿔서 계속 재사용할 수 있습니다.

### `{변수명}` 문법

- 중괄호 `{}` 안에 변수 이름을 씁니다
- `invoke({'변수명': '값'})`으로 변수를 채웁니다
- 변수가 채워진 결과물이 실제 AI에게 전달되는 프롬프트입니다

In [1]:
# .env 로드 + llm 초기화
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# 프로젝트 루트의 .env 파일 로드 (이 노트북은 02_prompts/ 폴더에 있음)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')

loaded = load_dotenv(env_path)

# API 키 확인
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 모듈 00을 먼저 완료하세요.')

# 모델 초기화
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.7,
)

print()
print('[OK] ChatGoogleGenerativeAI 초기화 완료!')
print(f'모델: {llm.model}')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...

[OK] ChatGoogleGenerativeAI 초기화 완료!
모델: gemini-2.5-flash


---

## 섹션 3: ChatPromptTemplate 기초

### ChatPromptTemplate 구조

`ChatPromptTemplate`은 `from_messages()` 메서드로 만듭니다. 각 메시지는 `(역할, 내용)` 튜플로 작성합니다:

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 전문가입니다."),  # 시스템 역할 설정
    ("human",  "{topic}에 대해 설명해주세요."),      # 사용자 질문
])
```

| 요소 | 설명 |
|------|------|
| `"system"` | AI에게 역할을 부여하는 메시지 (SystemMessage) |
| `"human"` | 사용자의 질문 메시지 (HumanMessage) |
| `{language}` | 나중에 채울 변수 - 중괄호로 표시 |
| `{topic}` | 나중에 채울 변수 - 중괄호로 표시 |

### 동작 방식

```
템플릿 정의 (변수 포함)  →  invoke()로 변수 채우기  →  완성된 메시지 리스트  →  llm 호출
```

In [2]:
# 기본 템플릿 생성 + 내부 구조 탐색
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {language} 전문가입니다. 초보자도 이해할 수 있게 설명하세요.'),
    ('human', '{topic}에 대해 설명해주세요.'),
])

print('=== 템플릿 내부 구조 확인 ===')                  
print(f'입력 변수: {prompt.input_variables}')
print(f'메시지 수: {len(prompt.messages)}')
for msg in prompt.messages:
    print(f'  [{msg.__class__.__name__}]: {msg.prompt.template}')

=== 템플릿 내부 구조 확인 ===
입력 변수: ['language', 'topic']
메시지 수: 2
  [SystemMessagePromptTemplate]: 당신은 {language} 전문가입니다. 초보자도 이해할 수 있게 설명하세요.
  [HumanMessagePromptTemplate]: {topic}에 대해 설명해주세요.


In [3]:
# invoke()로 변수 채우기 → 완성된 메시지 리스트 확인
filled = prompt.invoke({'language': 'Python', 'topic': '리스트 컴프리헨션'})

print('=== 변수 채운 후 메시지 확인 ===')               
print(f'메시지 수: {len(filled.messages)}')
print()
for msg in filled.messages:
    print(f'[{msg.type}]')
    print(f'  {msg.content}')

=== 변수 채운 후 메시지 확인 ===
메시지 수: 2

[system]
  당신은 Python 전문가입니다. 초보자도 이해할 수 있게 설명하세요.
[human]
  리스트 컴프리헨션에 대해 설명해주세요.


In [4]:
# 완성된 프롬프트로 llm 호출 → 응답 출력
print('=== Python 리스트 컴프리헨션 설명 ===')          
print()

response = llm.invoke(filled)
print(response.content)

=== Python 리스트 컴프리헨션 설명 ===

안녕하세요! 파이썬 리스트 컴프리헨션(List Comprehension)은 처음 들으면 좀 어렵게 느껴질 수 있지만, 한번 익숙해지면 파이썬 코드를 훨씬 **간결하고 멋지게** 만들 수 있는 아주 강력한 기능입니다. 초보자도 이해할 수 있도록 차근차근 설명해 드릴게요!

---

### 📦 리스트 컴프리헨션이란? (List Comprehension)

리스트 컴프리헨션은 **기존 리스트(또는 반복 가능한 객체)를 바탕으로 새로운 리스트를 만드는 파이썬의 특별한 문법**입니다. 여러 줄에 걸쳐 작성해야 했던 `for` 반복문과 `if` 조건문을 한 줄에 압축해서 쓸 수 있게 해줍니다.

쉽게 말해, "이 재료들 중에서, 이 조건에 맞는 것들만, 이렇게 요리해서 새 접시에 담아라!" 라고 한 문장으로 말하는 것과 같아요.

#### 왜 사용할까요? (장점)

1.  **간결함 (Conciseness):** 코드가 짧아지고 깔끔해집니다.
2.  **가독성 (Readability):** 익숙해지면 `for` 반복문보다 코드를 한눈에 이해하기 더 쉽습니다.
3.  **성능 (Performance):** 일반적으로 `for` 반복문으로 같은 작업을 하는 것보다 더 빠르게 동작합니다. (이건 나중에 더 깊이 배우실 때 중요하게 생각하시면 됩니다!)

---

### 🛠 기본 구조

리스트 컴프리헨션의 기본 구조는 다음과 같습니다:

```python
새로운_리스트 = [표현식 for 항목 in 이터러블 if 조건]
```

*   **`표현식 (Expression)`**: 새로운 리스트에 추가될 항목을 어떻게 만들지 정의합니다. (예: `item`, `item * 2`, `item.upper()`)
*   **`for 항목 in 이터러블 (for item in iterable)`**: 어떤 데이터(리스트, 튜플, 문자열, `range()` 등)에서 `항목`을 하나씩 꺼내올지 정의합니다.
*   **`if 조건 (if co

In [5]:
# 같은 템플릿, 다른 변수로 재사용 (Python / JavaScript / Java 비교)
languages = [
    ('Python', '리스트 컴프리헨션'),
    ('JavaScript', '화살표 함수'),
    ('Java', '제네릭'),
]

print('=== 같은 템플릿, 3개 언어 재사용 비교 ===')      
print()

for language, topic in languages:
    filled = prompt.invoke({'language': language, 'topic': topic})
    response = llm.invoke(filled)
    # 응답 첫 두 문장만 출력
    short = '. '.join(response.content.split('. ')[:2]) + '.'
    print(f'[{language}] {topic}')
    print(f'  {short}')
    print()

=== 같은 템플릿, 3개 언어 재사용 비교 ===

[Python] 리스트 컴프리헨션
  안녕하세요! Python 초보자분들도 리스트 컴프리헨션을 마스터할 수 있도록 아주 쉽고 친절하게 설명해 드릴게요.

---

## 🐍 리스트 컴프리헨션 (List Comprehension) 이란?

리스트 컴프리헨션은 파이썬에서 **새로운 리스트를 아주 빠르고 간결하게 만드는 특별한 방법**입니다. 마치 마법 상자처럼, 기존 리스트의 요소들을 가지고 원하는 대로 가공하거나 필터링해서 새로운 리스트를 뚝딱 만들어낼 수 있어요.

**핵심:**
1.

[JavaScript] 화살표 함수
  안녕하세요! 자바스크립트의 **화살표 함수(Arrow Function)**는 최신 자바스크립트(ES6)에서 도입된 기능으로, 함수를 만드는 또 다른 방법이자, 훨씬 더 간결하게 코드를 작성할 수 있도록 도와주는 멋진 도구입니다.

초보자도 이해하기 쉽게 비유를 들어 설명해 드릴게요.

---

### **함수란 무엇인가요? (간단 복습)**

자바스크립트에서 함수는 특정 작업을 수행하는 코드의 묶음입니다. 마치 **'미니 프로그램'**이나 **'요리 레시피'**와 같아요.
예를 들어, 두 숫자를 더하는 함수는 다음과 같이 만들 수 있습니다.

```javascript
// '두 숫자를 더하는 레시피'를 만드는 과정
function 더하기(숫자1, 숫자2) {
  return 숫자1 + 숫자2; // 숫자1과 숫자2를 더한 결과를 돌려줘
}

// 레시피를 사용해서 요리하기
let 결과 = 더하기(5, 3); // 5와 3을 더해줘!
console.log(결과); // 8
```

여기서 `function`이라는 키워드를 사용해서 함수를 만들었죠.

---

### **화살표 함수는 무엇인가요? (간결한 레시피!)**

화살표 함수는 이 '레시피'를 만드는 방법을 **더 짧고 간결하게** 만들어 줍니다.

[Java] 제네릭
  안녕하세요! Java의 '제네릭(Generics)'에 대해 초

---

## 섹션 4: partial() - 변수 미리 고정하기

### partial()이란?

**쿠키 틀 비유**

쿠키를 만들 때 반죽 재료(밀가루, 버터, 설탕)는 항상 같고, 모양만 별 모양/하트 모양으로 바꾼다고 생각해보세요.

매번 반죽 재료를 다시 넣을 필요 없이, **반죽은 미리 준비해두고 틀만 바꾸면** 됩니다.

`partial()`은 이처럼 자주 고정되는 변수를 미리 채워두는 기능입니다:

```python
# 원본 템플릿: language와 topic 두 변수
prompt = ChatPromptTemplate.from_messages([...])

# language를 'Python'으로 고정
python_prompt = prompt.partial(language='Python')

# 이제 topic만 채우면 됨
filled = python_prompt.invoke({'topic': '클래스'})
```

| 구분 | 원본 템플릿 | partial() 적용 후 |
|------|------------|------------------|
| 필요한 변수 | language, topic | topic (language는 고정) |
| 사용 상황 | 언어가 매번 달라질 때 | 언어가 항상 같은 맥락 |

In [6]:
# partial()로 language를 'Python'으로 고정 → topic만 바꿔서 호출
python_prompt = prompt.partial(language='Python')

print('=== partial() 적용 결과 ===')                    
print(f'원본 input_variables: {prompt.input_variables}')
print(f'partial 후 input_variables: {python_prompt.input_variables}')
print()

topics = ['클래스', '데코레이터', '제너레이터']

for topic in topics:
    filled = python_prompt.invoke({'topic': topic})
    response = llm.invoke(filled)
    short = response.content.split('\n')[0]
    print(f'[{topic}]')
    print(f'  {short}')
    print()

=== partial() 적용 결과 ===
원본 input_variables: ['language', 'topic']
partial 후 input_variables: ['topic']

[클래스]
  안녕하세요! 파이썬 클래스, 초보자도 쉽게 이해할 수 있도록 재미있는 비유와 함께 설명해 드릴게요. 너무 어렵게 생각하지 마세요!

[데코레이터]
  안녕하세요! 파이썬 데코레이터, 처음 들으면 마치 마법 주문처럼 느껴질 수 있지만, 알고 보면 정말 유용하고 재미있는 기능이랍니다. 초보자분들도 쉽게 이해할 수 있도록 차근차근 설명해 드릴게요.

[제너레이터]
  안녕하세요! 파이썬 제너레이터, 이름만 들으면 어려워 보이지만 사실은 아주 유용하고 재미있는 개념입니다. 초보자 눈높이에 맞춰 쉽고 친절하게 설명해 드릴게요!



---

## 섹션 5: FewShotChatMessagePromptTemplate

### Few-shot이란?

**시험 공부 비유**

수학 시험 공부를 할 때, 개념 설명만 읽는 것보다 **예제 문제를 몇 개 풀어보면** 패턴이 훨씬 빠르게 이해됩니다.

AI에게도 똑같습니다. 지시사항만 주는 것보다 **몇 가지 예시를 함께 주면** AI가 원하는 형식을 훨씬 잘 따라옵니다:

```
지시사항만: "감정을 분류하세요"         → AI가 어떤 형식으로 답할지 모름
Few-shot:   "입력: 맛있어! → 출력: 긍정"  → AI가 패턴을 보고 그대로 따라함
            "입력: 짜증나  → 출력: 부정"
            "입력: ???     → 출력: ?"
```

### 구성 요소

| 요소 | 역할 |
|------|------|
| `examples` | 예시 데이터 리스트 (딕셔너리 형태) |
| `example_prompt` | 예시 하나를 메시지로 변환하는 템플릿 |
| `FewShotChatMessagePromptTemplate` | 모든 예시를 묶어서 프롬프트로 만들기 |
| `final_prompt` | 시스템 메시지 + few-shot + 실제 질문 조합 |

In [7]:
# examples 데이터 정의 + example_prompt 생성 + 구조 확인
examples = [
    {'input': '오늘 점심이 정말 맛있었어!', 'output': '긍정'},
    {'input': '지하철이 또 연착됐어. 짜증나.', 'output': '부정'},
    {'input': '내일 회의가 있어.', 'output': '중립'},
    {'input': '드디어 프로젝트가 완성됐다!', 'output': '긍정'},
]

example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}'),
])

print('=== example_prompt 구조 확인 ===')               
print(f'입력 변수: {example_prompt.input_variables}')
print()
print('예시 0번을 채우면:')
filled = example_prompt.invoke(examples[0])
for msg in filled.messages:
    print(f'  [{msg.type}]: {msg.content}')

=== example_prompt 구조 확인 ===
입력 변수: ['input', 'output']

예시 0번을 채우면:
  [human]: 오늘 점심이 정말 맛있었어!
  [ai]: 긍정


### FewShotChatMessagePromptTemplate 구조

`FewShotChatMessagePromptTemplate`은 `examples`를 `example_prompt`로 변환해서 **모든 예시를 메시지로 펼쳐줍니다**.

이것을 `final_prompt`에 조합하면 전체 프롬프트가 완성됩니다:

```
final_prompt 구조:
┌────────────────────────────────────────────────────┐
│ [system] 텍스트의 감정을 분류하세요               │
│ [human]  오늘 점심이 정말 맛있었어!               │  ← examples[0]
│ [ai]     긍정                                     │  ← examples[0]
│ [human]  지하철이 또 연착됐어. 짜증나.            │  ← examples[1]
│ [ai]     부정                                     │  ← examples[1]
│ [human]  내일 회의가 있어.                        │  ← examples[2]
│ [ai]     중립                                     │  ← examples[2]
│ [human]  드디어 프로젝트가 완성됐다!              │  ← examples[3]
│ [ai]     긍정                                     │  ← examples[3]
│ [human]  {input}  ← 실제 분류할 문장             │
└────────────────────────────────────────────────────┘
```

AI는 이 패턴을 보고 `{input}`에 대해 '긍정'/'부정'/'중립' 중 하나만 답합니다.

In [8]:
# FewShotChatMessagePromptTemplate 생성 + final_prompt 조합 + 메시지 확인
from langchain_core.prompts import FewShotChatMessagePromptTemplate

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', "텍스트의 감정을 '긍정', '부정', '중립' 중 하나로만 답하세요."),
    few_shot_prompt,
    ('human', '{input}'),
])

# 완성된 메시지 수 확인
filled = final_prompt.invoke({'input': '테스트'})
print(f'총 메시지 수: {len(filled.messages)}')
print(f'  - system 1개 + 예시 {len(examples)}개 x 2 + human 1개 = {1 + len(examples)*2 + 1}개')
print()
print('메시지 목록:')
for msg in filled.messages:
    print(f'  [{msg.type}]: {msg.content}')

총 메시지 수: 10
  - system 1개 + 예시 4개 x 2 + human 1개 = 10개

메시지 목록:
  [system]: 텍스트의 감정을 '긍정', '부정', '중립' 중 하나로만 답하세요.
  [human]: 오늘 점심이 정말 맛있었어!
  [ai]: 긍정
  [human]: 지하철이 또 연착됐어. 짜증나.
  [ai]: 부정
  [human]: 내일 회의가 있어.
  [ai]: 중립
  [human]: 드디어 프로젝트가 완성됐다!
  [ai]: 긍정
  [human]: 테스트


In [9]:
# 감정 분류 실행 (여러 문장 테스트 + [OK]/결과 출력)
# few-shot 감정 분류는 temperature=0.3으로 일관성 높게
llm_precise = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3,
)

test_sentences = [
    ('코드 리뷰가 생각보다 빨리 끝났어.', '긍정'),
    ('버그가 너무 많아서 힘들다.', '부정'),
    ('오늘 스탠드업 미팅 10시야.', '중립'),
    ('드디어 배포 성공!', '긍정'),
    ('왜 이게 또 안 되는 거야...', '부정'),
]

print('=== 감정 분류 실행 ===')                         
print()

correct = 0
for sentence, expected in test_sentences:
    filled = final_prompt.invoke({'input': sentence})
    response = llm_precise.invoke(filled)
    result = response.content.strip()
    status = '[OK]  ' if result == expected else '[FAIL]'
    if result == expected:
        correct += 1
    print(f'{status} 입력: {sentence}')
    print(f'       결과: {result} (예상: {expected})')
    print()

print(f'정확도: {correct}/{len(test_sentences)}')

=== 감정 분류 실행 ===

[OK]   입력: 코드 리뷰가 생각보다 빨리 끝났어.
       결과: 긍정 (예상: 긍정)

[OK]   입력: 버그가 너무 많아서 힘들다.
       결과: 부정 (예상: 부정)

[OK]   입력: 오늘 스탠드업 미팅 10시야.
       결과: 중립 (예상: 중립)

[OK]   입력: 드디어 배포 성공!
       결과: 긍정 (예상: 긍정)

[OK]   입력: 왜 이게 또 안 되는 거야...
       결과: 부정 (예상: 부정)

정확도: 5/5


---

## 섹션 6: 실전 응용

In [10]:
# 번역기 만들기 (partial로 target_language 고정, 입력만 바꾸기)
translator_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 전문 번역가입니다. 입력된 텍스트를 {target_language}로 번역하세요. 번역문만 출력하세요.'),
    ('human', '{text}'),
])

# target_language를 '영어'로 고정한 번역기
english_translator = translator_prompt.partial(target_language='영어')
# target_language를 '일본어'로 고정한 번역기
japanese_translator = translator_prompt.partial(target_language='일본어')

print('=== 번역기 실행 ===')                            
print(f'원본 변수: {translator_prompt.input_variables}')
print(f'english_translator 변수: {english_translator.input_variables}')
print(f'japanese_translator 변수: {japanese_translator.input_variables}')
print()

texts = [
    '인공지능은 미래의 핵심 기술입니다.',
    'LangChain으로 AI 앱을 쉽게 만들 수 있습니다.',
]

for text in texts:
    en = llm.invoke(english_translator.invoke({'text': text})).content.strip()
    ja = llm.invoke(japanese_translator.invoke({'text': text})).content.strip()
    print(f'[원문] {text}')
    print(f'[영어] {en}')
    print(f'[일본어] {ja}')
    print()

=== 번역기 실행 ===
원본 변수: ['target_language', 'text']
english_translator 변수: ['text']
japanese_translator 변수: ['text']

[원문] 인공지능은 미래의 핵심 기술입니다.
[영어] Artificial Intelligence is a core technology for the future.
[일본어] 人工知能は未来の中核技術です。

[원문] LangChain으로 AI 앱을 쉽게 만들 수 있습니다.
[영어] You can easily create AI apps with LangChain.
[일본어] LangChainでAIアプリを簡単に作成できます。



---

## 섹션 7: 마무리

### 배운 것 정리

#### ChatPromptTemplate vs FewShotChatMessagePromptTemplate 비교

| 구분 | ChatPromptTemplate | FewShotChatMessagePromptTemplate |
|------|-------------------|----------------------------------|
| 용도 | 변수가 있는 재사용 가능한 프롬프트 | 예시를 주어 패턴을 따르게 하는 프롬프트 |
| 구성 | system + human 메시지 | examples 리스트 → 메시지로 변환 |
| 변수 문법 | `{변수명}` | `{input}`, `{output}` |
| 사용 상황 | 역할/형식 지정이 필요할 때 | 출력 형식을 예시로 보여줄 때 |

#### 핵심 코드 패턴

```python
# 1. 기본 템플릿
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {role}입니다.'),
    ('human', '{question}'),
])
filled = prompt.invoke({'role': '번역가', 'question': '안녕하세요를 영어로'})
response = llm.invoke(filled)

# 2. partial() - 일부 변수 미리 고정
translator = prompt.partial(role='영어 번역가')
filled = translator.invoke({'question': '안녕하세요를 영어로'})  # role은 자동 채워짐

# 3. Few-shot - 예시를 보여주고 패턴 따르게 하기
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,  # 예시 하나를 메시지로 변환
    examples=examples,              # 예시 데이터 리스트
)
final = ChatPromptTemplate.from_messages([
    ('system', '지시사항'),
    few_shot,                       # 예시 메시지들이 여기 들어감
    ('human', '{input}'),           # 실제 질문
])
```

## 다음 모듈 예고: 모듈 03 - LCEL: | 파이프 연산자

모듈 02에서는 프롬프트를 채우고 → llm에 전달하는 두 단계를 직접 연결했습니다.

```python
# 지금까지 방식
filled = prompt.invoke({'language': 'Python', 'topic': '클래스'})
response = llm.invoke(filled)
```

다음 모듈에서는 이 두 단계를 `|` 파이프 연산자 하나로 연결하는 **LCEL(LangChain Expression Language)**을 배웁니다:

```python
# 모듈 03: LCEL 방식
chain = prompt | llm
response = chain.invoke({'language': 'Python', 'topic': '클래스'})
```

### 준비 체크리스트

다음 모듈로 넘어가기 전에 확인하세요:

- [ ] 셀 9: 변수를 채운 프롬프트로 Gemini 응답이 출력되었나요?
- [ ] 셀 10: 3개 언어로 같은 템플릿 재사용이 동작했나요?
- [ ] 셀 12: partial()로 language 고정 후 topic만 바꿔서 호출이 동작했나요?
- [ ] 셀 17: 감정 분류가 '긍정'/'부정'/'중립'으로 출력되었나요?
- [ ] 셀 18: 번역기 실행 결과가 출력되었나요?

모두 체크했다면 `03_lcel/모듈03_LCEL_파이프_연산자.ipynb`를 열어보세요!

---

수고하셨습니다!